<a href="https://colab.research.google.com/github/litlig/notebooks/blob/main/discrete_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Build language generation model using discrete diffusion model

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-14 18:13:08--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-07-14 18:13:08 (31.8 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print("length of dataset in characters: ", len(text))
print(text[:200])

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [3]:
vocab = sorted(list(set(text)))
mask = '+'
vocab.insert(0, mask)
vocab_size = len(vocab)
print("vocab size: ", len(vocab))
print("vocab: ", vocab)

stoi = { ch:i for i,ch in enumerate(vocab) }
itos = { i:ch for i,ch in enumerate(vocab) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


vocab size:  66
vocab:  ['+', '\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
import torch

if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
data = torch.tensor(encode(text), dtype=torch.long, device=device)

In [5]:
import torch
import torch.nn.functional as F

from torch.distributions import Bernoulli

ivocab = torch.tensor(list(itos.keys()), dtype=torch.long)

n_sample = 32
n_seq = 64

def schedule(t):
  return t

def get_batch():
  t = torch.rand(n_sample, device=device)
  kappa = schedule(t)
  seq_idx = torch.randint(0, len(text) - n_seq + 1, (n_sample,), device=device).unsqueeze(1) + torch.arange(n_seq, device=device)
  z = data[seq_idx] # sample:seq
  masks = torch.bernoulli(kappa.unsqueeze(1).expand(-1, n_seq)).long() # sample:seq
  # noise_idx = torch.randint(0, len(vocab), (n_sample, n_seq))
  # noise = ivocab[noise_idx] # sample:seq
  noise = torch.zeros_like(z, device=device)
  x = masks * z + (1-masks) * noise
  return x, t, z

def loss_fn(z_pred, z):
  return F.cross_entropy(z_pred.view(-1, vocab_size), z.view(-1))


In [6]:
#@title model arch
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class SinusoidalTimeEmbedding(nn.Module):
    """Standard transformer-style sinusoidal embedding for a scalar t."""

    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device).float() / half
        )
        args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)  # (B, half)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)
        return emb

class TimestepMLP(nn.Module):
    """Sinusoidal embedding -> MLP -> conditioning vector c."""

    def __init__(self, hidden_dim: int, cond_dim: int):
        super().__init__()
        self.sinusoidal = SinusoidalTimeEmbedding(hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, cond_dim),
            nn.SiLU(),
            nn.Linear(cond_dim, cond_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.mlp(self.sinusoidal(t))  # (B, cond_dim)


def modulate(x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class AdaLNZeroBlock(nn.Module):
    """
    Transformer encoder block with AdaLN-Zero conditioning on t.

    Six modulation params per block (as in DiT):
      shift_msa, scale_msa, gate_msa   -> for the attention sub-layer
      shift_mlp, scale_mlp, gate_mlp   -> for the MLP sub-layer

    The final Linear producing these params is zero-initialized so each
    block starts as an identity function (gate = 0), which stabilizes
    training at initialization.
    """

    def __init__(self, dim: int, num_heads: int, cond_dim: int, mlp_ratio: float = 4.0,
                 dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 6 * dim),
        )
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x: torch.Tensor, c: torch.Tensor,
                attn_mask: torch.Tensor = None,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        (shift_msa, scale_msa, gate_msa,
         shift_mlp, scale_mlp, gate_mlp) = self.adaLN_modulation(c).chunk(6, dim=-1)

        h = modulate(self.norm1(x), shift_msa, scale_msa)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        x = x + gate_msa.unsqueeze(1) * attn_out

        h = modulate(self.norm2(x), shift_mlp, scale_mlp)
        x = x + gate_mlp.unsqueeze(1) * self.mlp(h)
        return x


class FinalAdaLN(nn.Module):
    """Final norm + modulation before projecting to vocab logits."""

    def __init__(self, dim: int, cond_dim: int, vocab_size: int):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 2 * dim),
        )
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)
        self.head = nn.Linear(dim, vocab_size)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=-1)
        x = modulate(self.norm(x), shift, scale)
        return self.head(x)  # (B, L, vocab_size)

class DiscreteDiffusionTransformer(nn.Module):
    """
    Input:  x_t (B, L) token ids (possibly containing a [MASK] id), t (B,)
    Output: logits (B, L, vocab_size) predicting the denoised/original tokens
    """

    def __init__(
        self,
        vocab_size: int,
        max_seq_len: int = 64,
        dim: int = 64,
        depth: int = 4,
        num_heads: int = 4,
        cond_dim: int = 64,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_seq_len, dim))
        nn.init.normal_(self.pos_emb, std=0.02)

        self.time_mlp = TimestepMLP(hidden_dim=cond_dim, cond_dim=cond_dim)

        self.blocks = nn.ModuleList([
            AdaLNZeroBlock(dim, num_heads, cond_dim, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.final = FinalAdaLN(dim, cond_dim, vocab_size)

    def forward(
        self,
        x_t: torch.Tensor,          # (B, L) long
        t: torch.Tensor,            # (B,) float or long
        padding_mask: torch.Tensor = None,  # (B, L) bool, True = PAD (ignored)
    ) -> torch.Tensor:
        B, L = x_t.shape
        x = self.token_emb(x_t) + self.pos_emb[:, :L, :]
        c = self.time_mlp(t)  # (B, cond_dim) — conditioning vector shared across layers

        for block in self.blocks:
            x = block(x, c, key_padding_mask=padding_mask)

        logits = self.final(x, c)
        return logits

In [8]:
model = DiscreteDiffusionTransformer(vocab_size).to(device)

optimizer = torch.optim.Adam(model.parameters(), 1e-3)
losses = []
num_epochs = 1000

print(device)
for epoch in range(num_epochs):
  optimizer.zero_grad()
  x, t, z = get_batch()
  padding_mask = x == 0

  logits = model(x, t, padding_mask)
  loss = loss_fn(logits, z)
  loss.backward()
  optimizer.step()
  losses.append(loss.item())

  if epoch % 100 == 0:
    print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

cuda
Epoch 0, Loss: 4.189655
Epoch 100, Loss: 1.581045
Epoch 200, Loss: 1.392731
Epoch 300, Loss: 1.681531
Epoch 400, Loss: 1.731939
Epoch 500, Loss: 1.700989
Epoch 600, Loss: 1.586851
Epoch 700, Loss: 2.067987
Epoch 800, Loss: 1.672981
Epoch 900, Loss: 1.647514
